# Case 9 -- Two-Fluid, Transient, Vertical Flow



In [ ]:
from IPython.display import HTML
import pathlib as pl
import numpy as np
import matplotlib.pyplot as plt
import flopy
from swiutil import SwiAnimator

# path to mf6 executables with swi support: 
#   https://github.com/christianlangevin/modflow6-nightly-build/actions/workflows/nightly-build-swi.yml

# Put the name of the mf6 executable into mf6exe.txt,
# which is not under version control.
with open(pl.Path("./mf6exe.txt"), "r") as f:
    mf6exe = f.readline().strip()
print(f"using executable: {mf6exe}")

sim_ws = pl.Path("./temp/case9")

In [ ]:
def calc_zeta(hf, hs):
    alphaf = 1000 / (1025 - 1000)
    alphas = 1025 / (1025 - 1000)
    zeta = -alphaf * hf + alphas * hs
    return zeta

calc_zeta(0.0125, 0.0)

In [ ]:
ncol = 1
Lx = 1
dx = 1.0
nlay = 5
nrow = 1
delr = np.array(ncol * [dx])

delc = 1.0
botm = [-float(k + 1) for k in range(nlay)]
rate = 0.1
recharge = {0: rate, 1: -rate}
k_fw = 1.0
k_sw = 1.0
h0 = 0.0
icelltype = 0
iconvert = 0
top = 0.0
newtonoptions = "NEWTON"
ss = 0.0
sy = 0.1
inner_dvclose = 1.e-8
outer_dvclose = 1.e-7
outer_maximum = 500
inner_maximum = 100
# inner_dvclose = 1.e-5
# outer_dvclose = 1.e-4

perioddata = [(1.0, 10, 1.0), (1.0, 10, 1.0)]
# perioddata = [(3.0, 10, 1.0)]


def build_gwf_model(sim, is_saltwater):
    if is_saltwater:
        name = "saltwater"
    else:
        name = "freshwater"

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=name,
        save_flows=True,
        newtonoptions=newtonoptions,
    )
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
    )

    strt_fw = nlay * [0.0125]
    strt_sw = nlay * [0.0]
    strt = strt_sw if is_saltwater else strt_fw
    ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        save_saturation=True,
        # alternative_cell_averaging=None,
        icelltype=icelltype,
        k=k_sw if is_saltwater else k_fw,
    )
    sto = flopy.mf6.ModflowGwfsto(gwf, iconvert=iconvert, ss=ss, sy=sy)
    zeta_file = name + ".zta"
    swi = flopy.mf6.ModflowGwfswi(
        gwf,
        zeta_filerecord=zeta_file,
    )
    if is_saltwater:
        # single CHD at the bottom of model for outflow
        # chd = flopy.mf6.ModflowGwfchd(
        #     gwf,
        #     stress_period_data=[[nlay - 1, 0, 0, h0]],
        # )

        # single GHB at the bottom of model for outflow
        ghb = flopy.mf6.ModflowGwfghb(
            gwf,
            stress_period_data=[[nlay - 1, 0, 0, h0, 10.0]],
        )

    if not is_saltwater:
        rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)

    budget_file = name + ".bud"
    head_file = name + ".hds"
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )

    return gwf


def build_models():
    ws = sim_ws
    sim_name = "mymodel"
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=ws,
        exe_name=mf6exe,
        memory_print_option="all",
        # continue_=True,
        # print_input=True,
    )

    # transient tdis
    nper = len(perioddata)
    tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)

    ims = flopy.mf6.ModflowIms(
        sim,
        complexity="complex",
        print_option="summary",
        no_ptcrecord=True,
        under_relaxation="DBD",
        under_relaxation_gamma=0.1,
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.07,
        under_relaxation_momentum=0.0,
        outer_maximum=outer_maximum,
        inner_maximum=inner_maximum,
        outer_dvclose=outer_dvclose,
        inner_dvclose=inner_dvclose,
        linear_acceleration="bicgstab",
        preconditioner_levels=7,
        number_orthogonalizations=14,
        preconditioner_drop_tolerance=1e-3,
    )

    gwf_freshwater = build_gwf_model(sim, False)
    gwf_saltwater = build_gwf_model(sim, True)

    swiswi = flopy.mf6.ModflowSwiswi(
        sim,
        print_input=True,
        print_flows=True,
        exgtype="SWI6-SWI6",
        exgmnamea="freshwater",
        exgmnameb="saltwater",
    )
    sim.register_ims_package(ims, [gwf_freshwater.name, gwf_saltwater.name])

    return sim


def plot_cell_patch(top, bot, zeta):
    dx = 1.0
    from matplotlib.patches import Rectangle
    r_sw = Rectangle((0, bot), dx, max(zeta - bot, 0.), facecolor="red", edgecolor="k")
    r_fw = Rectangle((0, zeta), dx, top - zeta, facecolor="blue", edgecolor="k")
    return r_fw, r_sw

def plot_grid_patches(ax, hobj, zobj, times, tidx):
    top = np.linspace(0, -4, 5)
    bot = np.linspace(-1, -5, 5)
    zeta = zobj.get_data(totim=times[tidx]).flatten()
    for k in range(nlay):
        r_fw, r_sw = plot_cell_patch(top[k], bot[k], zeta[k])
        ax.add_patch(r_fw)
        ax.add_patch(r_sw)
    return ax

def plot_output(sim):
    import matplotlib.pyplot as plt

    ws = sim_ws
    gwf = sim.gwf[0]
    x = gwf.modelgrid.xcellcenters.flatten()
    z = np.linspace(-0.5, -4.5, nlay)
    fpth = pl.Path(ws) / f"{gwf.name}.zta"
    hobj = gwf.output.head()
    zobj = flopy.utils.HeadFile(fpth, text="zeta")
    times = zobj.times

    f = plt.figure(figsize=(10, 6))
    num_plots = len(times)
    for tidx in range(num_plots):
        ax = f.add_subplot(1, num_plots, tidx + 1)
        ax = plot_grid_patches(ax, hobj, zobj, times, tidx)
        ax.set_ylim(-5, 0.0)
        ax.set_xlim(0, 1.0)
        ax.set_aspect("equal")
        ax.set_title(f"t={times[tidx]:0.2f}")
        if tidx > 0:
            ax.set_yticklabels([])


In [ ]:
sim = build_models()
sim.write_simulation()
sim.run_simulation()
plot_output(sim)

In [ ]:
animator = SwiAnimator(sim=sim)
ani = animator.create()
HTML(ani.to_jshtml())

## Single-Fluid, Transient, Vertical Flow

Updated to include a recharge period followed by a withdrawal period

In [ ]:
ncol = 1
Lx = 1
dx = 1.0
nlay = 5
nrow = 1
delr = np.array(ncol * [dx])

delc = 1.0
botm = [-float(k + 1) for k in range(nlay)]
rate = 0.1
recharge = {0: rate, 1: -rate}
k_fw = 1.0
k_sw = 1.0
h0 = 0.0
icelltype = 1
iconvert = 1
top = 0.0
newtonoptions = "NEWTON"
ss = 0.0
sy = 0.1
inner_dvclose = 1.e-8
outer_dvclose = 1.e-7
# inner_dvclose = 1.e-4
# outer_dvclose = 1.e-3

perioddata = [(2.0, 10, 1.0), (2.0, 10, 1.0)]


def build_gwf_model(sim, is_saltwater):
    if is_saltwater:
        name = "saltwater"
    else:
        name = "freshwater"

    gwf = flopy.mf6.ModflowGwf(
        sim,
        modelname=name,
        save_flows=True,
        newtonoptions=newtonoptions,
    )
    dis = flopy.mf6.ModflowGwfdis(
        gwf,
        nlay=nlay,
        nrow=nrow,
        ncol=ncol,
        delr=delr,
        delc=delc,
        top=top,
        botm=botm,
    )

    strt_fw = nlay * [0.0125]
    strt_sw = nlay * [0.0]
    strt = strt_sw if is_saltwater else strt_fw
    ic = flopy.mf6.ModflowGwfic(gwf, strt=strt)
    npf = flopy.mf6.ModflowGwfnpf(
        gwf,
        save_specific_discharge=True,
        save_saturation=True,
        # alternative_cell_averaging=None,
        icelltype=icelltype,
        k=k_sw if is_saltwater else k_fw,
    )
    sto = flopy.mf6.ModflowGwfsto(gwf, iconvert=iconvert, ss=ss, sy=sy)
    zeta_file = name + ".zta"
    swi = flopy.mf6.ModflowGwfswi(
        gwf,
        zeta_filerecord=zeta_file,
    )
    if is_saltwater:
        # single CHD at the bottom of model for outflow
        chd = flopy.mf6.ModflowGwfchd(
            gwf,
            stress_period_data=[[nlay - 1, 0, 0, h0]],
        )
    if not is_saltwater:
        rch = flopy.mf6.ModflowGwfrcha(gwf, recharge=recharge)

    budget_file = name + ".bud"
    head_file = name + ".hds"
    oc = flopy.mf6.ModflowGwfoc(
        gwf,
        budget_filerecord=budget_file,
        head_filerecord=head_file,
        saverecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
        printrecord=[("HEAD", "ALL"), ("BUDGET", "ALL")],
    )

    return gwf


def build_models():
    ws = sim_ws
    sim_name = "mymodel"
    sim = flopy.mf6.MFSimulation(
        sim_name=sim_name,
        sim_ws=ws,
        exe_name=mf6exe,
        memory_print_option="all",
        # print_input=True,
    )

    # transient tdis
    nper = len(perioddata)
    tdis = flopy.mf6.ModflowTdis(sim, nper=nper, perioddata=perioddata)

    ims = flopy.mf6.ModflowIms(
        sim,
        print_option="summary",
        no_ptcrecord=True,
        under_relaxation="DBD",
        under_relaxation_gamma=0.1,
        under_relaxation_theta=0.7,
        under_relaxation_kappa=0.07,
        under_relaxation_momentum=0.0,
        outer_maximum=500,
        inner_maximum=600,
        outer_dvclose=outer_dvclose,
        inner_dvclose=inner_dvclose,
        linear_acceleration="bicgstab",
        preconditioner_levels=7,
        number_orthogonalizations=14,
        preconditioner_drop_tolerance=1e-3,
    )

    gwf_freshwater = build_gwf_model(sim, False)
    # gwf_saltwater = build_gwf_model(sim, True)

    # swiswi = flopy.mf6.ModflowSwiswi(
    #     sim,
    #     print_input=True,
    #     print_flows=True,
    #     exgtype="SWI6-SWI6",
    #     exgmnamea="freshwater",
    #     exgmnameb="saltwater",
    # )
    # sim.register_ims_package(ims, [gwf_freshwater.name, gwf_saltwater.name])

    return sim


def plot_cell_patch(top, bot, zeta):
    dx = 1.0
    from matplotlib.patches import Rectangle
    r_sw = Rectangle((0, bot), dx, max(zeta - bot, 0.), facecolor="red", edgecolor="k")
    r_fw = Rectangle((0, zeta), dx, top - zeta, facecolor="blue", edgecolor="k")
    return r_fw, r_sw

def plot_grid_patches(ax, hobj, zobj, times, tidx):
    top = np.linspace(0, -4, 5)
    bot = np.linspace(-1, -5, 5)
    zeta = zobj.get_data(totim=times[tidx]).flatten()
    for k in range(nlay):
        r_fw, r_sw = plot_cell_patch(top[k], bot[k], zeta[k])
        ax.add_patch(r_fw)
        ax.add_patch(r_sw)
    return ax

def plot_output(sim):
    import matplotlib.pyplot as plt

    ws = sim_ws
    gwf = sim.gwf[0]
    x = gwf.modelgrid.xcellcenters.flatten()
    z = np.linspace(-0.5, -4.5, nlay)
    fpth = pl.Path(ws) / f"{gwf.name}.zta"
    hobj = gwf.output.head()
    zobj = flopy.utils.HeadFile(fpth, text="zeta")
    times = zobj.times

    f = plt.figure(figsize=(12, 8))
    num_plots = len(times)
    for tidx in range(num_plots):
        ax = f.add_subplot(1, num_plots, tidx + 1)
        ax = plot_grid_patches(ax, hobj, zobj, times, tidx)
        ax.set_ylim(-5, 0.0)
        ax.set_xlim(0, 1.0)
        ax.set_aspect("equal")
        ax.set_title(f"t={times[tidx]:0.1f}")
        if tidx > 0:
            ax.set_yticklabels([])

sim = build_models()
sim.write_simulation()
sim.run_simulation()
plot_output(sim)

In [ ]:
animator = SwiAnimator(sim=sim)
ani = animator.create()
HTML(ani.to_jshtml())